# Practice 6 — Data Transformation & EDA (SQLAlchemy + Pandas)

**Author:** Kenzhebay Ayala  
**Database:** SQLite (`hotel.db`) with two related tables: `rooms` and `bookings` (a small hotel booking database).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, inspect, text

## Preparation: create the small database
The database has two related tables (`bookings.room_id` -> `rooms.room_id`) with numerical and categorical columns.
A few missing values are added on purpose so that Task 3 has something to process.

In [ ]:
np.random.seed(42)

# table 1: rooms
rooms = pd.DataFrame({
    "room_id": range(1, 13),
    "room_type": ["Single"] * 4 + ["Double"] * 5 + ["Suite"] * 3,
    "price_per_night": [40, 40, 45, 45, 70, 70, 75, 80, 80, 150, 160, 180],
    "floor": [1, 1, 2, 2, 1, 2, 2, 3, 3, 4, 4, 5],
})

# table 2: bookings
n = 80
bookings = pd.DataFrame({
    "booking_id": range(1, n + 1),
    "room_id": np.random.randint(1, 13, n),
    "guest_name": [f"Guest {i}" for i in range(1, n + 1)],
    "check_in": pd.date_range("2026-01-01", periods=n, freq="3D").strftime("%Y-%m-%d"),
    "nights": np.random.randint(1, 8, n),
    "guests": np.random.randint(1, 5, n).astype(float),
    "status": np.random.choice(["Completed", "Confirmed", "Cancelled"], n, p=[0.6, 0.2, 0.2]),
})
bookings.loc[[5, 17, 33, 48, 60], "guests"] = np.nan      # missing values

engine = create_engine("sqlite:///hotel.db")
with engine.begin() as conn:
    rooms.to_sql("rooms", conn, if_exists="replace", index=False)
    bookings.to_sql("bookings", conn, if_exists="replace", index=False)
print("Database created")

## Task 1. Database connection

In [ ]:
engine = create_engine("sqlite:///hotel.db")

# available tables
print("Tables:", inspect(engine).get_table_names())

# load one table into a DataFrame
with engine.connect() as conn:
    df_bookings = pd.read_sql(text("SELECT * FROM bookings"), conn)

print("Rows:", df_bookings.shape[0], " Columns:", df_bookings.shape[1])
print("Column names:", list(df_bookings.columns))
print(df_bookings.dtypes)
df_bookings.head()

## Task 2. SQL queries

In [ ]:
# 1) WHERE: completed bookings with 5 or more nights
q1 = "SELECT * FROM bookings WHERE status = 'Completed' AND nights >= 5"

# 2) ORDER BY: the 10 latest bookings
q2 = "SELECT booking_id, guest_name, check_in, nights FROM bookings ORDER BY check_in DESC LIMIT 10"

# 3) GROUP BY: number of bookings and average nights per status
q3 = '''
SELECT status, COUNT(*) AS bookings_count, ROUND(AVG(nights), 2) AS avg_nights
FROM bookings
GROUP BY status
'''

# 4) JOIN: bookings + rooms
q4 = '''
SELECT b.booking_id, b.guest_name, b.check_in, b.nights, b.guests, b.status,
       r.room_type, r.price_per_night
FROM bookings AS b
JOIN rooms AS r ON b.room_id = r.room_id
'''

with engine.connect() as conn:
    df_where = pd.read_sql(text(q1), conn)
    df_order = pd.read_sql(text(q2), conn)
    df_group = pd.read_sql(text(q3), conn)
    df_join = pd.read_sql(text(q4), conn)

print("WHERE:", df_where.shape)
print("ORDER BY:"); display(df_order.head())
print("GROUP BY:"); display(df_group)
print("JOIN:", df_join.shape); display(df_join.head())

## Task 3. Data transformation
We use the JOIN result (`df_join`).

In [ ]:
df = df_join.copy()
print("Missing values before:")
print(df.isna().sum())

In [ ]:
# 1. missing values: unknown number of guests -> fill with 1 (at least one guest)
df["guests"] = df["guests"].fillna(1)

# 2. data types: guests float -> int, check_in text -> datetime
df["guests"] = df["guests"].astype(int)
df["check_in"] = pd.to_datetime(df["check_in"])

# 3. new columns
df["total_price"] = df["nights"] * df["price_per_night"]
df["month"] = df["check_in"].dt.month

# 4. remove unnecessary records and columns: cancelled bookings, guest name
df = df[df["status"] != "Cancelled"]
df = df.drop(columns=["guest_name"])

print("Missing values after:", df.isna().sum().sum())
print(df.dtypes)
df.head()

## Task 4. ETL pipeline
One Python script (`etl_pipeline.py`) does the whole process: **source data (CSV) → processed DataFrame → database table**.

In [ ]:
# the source file for ETL: the JOIN result saved as CSV
df_join.to_csv("bookings_source.csv", index=False)

In [ ]:
%%writefile etl_pipeline.py
import pandas as pd
from sqlalchemy import create_engine, text

# ---------- 1. EXTRACT ----------
df = pd.read_csv("bookings_source.csv")
print("Extracted rows:", len(df))

# ---------- 2. TRANSFORM ----------
df = df[["booking_id", "check_in", "nights", "guests", "status", "room_type", "price_per_night"]].copy()  # select columns
df["guests"] = df["guests"].fillna(1).astype(int)                 # missing values + data type
df["check_in"] = pd.to_datetime(df["check_in"])                   # data type
df = df.rename(columns={"price_per_night": "price"})              # rename column
df = df[df["status"] != "Cancelled"]                              # filter rows
df["total_price"] = df["nights"] * df["price"]                    # new column
print("Transformed rows:", len(df))

# ---------- 3. LOAD ----------
engine = create_engine("sqlite:///hotel.db")
with engine.begin() as conn:
    df.to_sql("bookings_clean", conn, if_exists="replace", index=False)
print("Loaded into table bookings_clean")

# ---------- 4. CHECK: read the table back ----------
with engine.connect() as conn:
    check = pd.read_sql(text("SELECT * FROM bookings_clean LIMIT 5"), conn)
print(check)

In [ ]:
!python etl_pipeline.py

## Task 5. Data check

In [ ]:
with engine.connect() as conn:
    loaded = pd.read_sql(text("SELECT * FROM bookings_clean"), conn)

expected_rows = (df_join["status"] != "Cancelled").sum()

print("Rows in database table:", len(loaded), "| expected:", expected_rows)
print("Missing values:", loaded.isna().sum().sum())
print("Duplicate booking_id:", loaded["booking_id"].duplicated().sum())
print("Cancelled rows left:", (loaded["status"] == "Cancelled").sum())
print("total_price is correct:", (loaded["total_price"] == loaded["nights"] * loaded["price"]).all())
print(loaded.dtypes)

assert len(loaded) == expected_rows
assert loaded.isna().sum().sum() == 0
print("\nETL check passed")
loaded.head()

## Extra: GroupBy, pivot / melt and plots (EDA)
Short exploration of the cleaned data from Task 3.

In [ ]:
# GroupBy: revenue by room type
revenue = df.groupby("room_type")["total_price"].agg(["count", "sum", "mean"]).round(1)
revenue

In [ ]:
# pivot: total price by month (rows) and room type (columns)
pivot = df.pivot_table(index="month", columns="room_type", values="total_price", aggfunc="sum", fill_value=0)
display(pivot)

# melt: back to the long format
long = pivot.reset_index().melt(id_vars="month", var_name="room_type", value_name="total_price")
long.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.barplot(data=df, x="room_type", y="total_price", estimator="sum", errorbar=None, ax=axes[0])
axes[0].set_title("Total revenue by room type")

sns.histplot(data=df, x="nights", bins=7, ax=axes[1])
axes[1].set_title("Distribution of nights")

sns.lineplot(data=long, x="month", y="total_price", hue="room_type", marker="o", ax=axes[2])
axes[2].set_title("Revenue by month")

plt.tight_layout()
plt.show()

## Conclusion
- Connected to SQLite with SQLAlchemy and loaded tables into Pandas.
- Used WHERE, ORDER BY, GROUP BY and JOIN queries.
- Cleaned the JOIN result (missing values, data types, new columns, removed cancelled records).
- Built an ETL script (CSV → DataFrame → database table) and checked the loaded table.